# 🧠 Prime Number Prediction with a Transformer

This notebook trains an improved transformer model to find patterns in the sequence of prime numbers. 

**Key Improvements:**
1.  **Predicting Gaps:** Instead of predicting the next prime, we predict the *gap* to the next prime, which is a more stable learning target.
2.  **GPU Acceleration:** The notebook is designed to automatically use a GPU if one is available.
3.  **Modern Architecture:** Uses sinusoidal positional encoding, a standard best practice for transformers.

--- 
### **Instructions: Enable the GPU**
1. Go to **Runtime** in the menu bar.
2. Select **Change runtime type**.
3. Choose **T4 GPU** from the *Hardware accelerator* dropdown.
4. Click **Save**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random
from tqdm.notebook import tqdm

# Check for GPU and set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Prime Generation and Data Preparation

First, we'll generate a large list of primes. Then, we'll transform this list into input sequences and target *gaps*.

In [ ]:
def generate_primes(n):
    """Generates a list of primes up to n using a sieve."""
    primes = []
    sieve = [True] * (n + 1)
    for p in range(2, n + 1):
        if sieve[p]:
            primes.append(p)
            for i in range(p * p, n + 1, p):
                sieve[i] = False
    return primes

def create_gap_sequences(primes, seq_length):
    """Create input sequences of primes and target gaps."""
    sequences = []
    for i in range(len(primes) - seq_length - 1):
        input_seq = primes[i:i + seq_length]
        last_prime_in_seq = input_seq[-1]
        target_prime = primes[i + seq_length]
        gap = target_prime - last_prime_in_seq
        sequences.append((torch.tensor(input_seq, dtype=torch.float32), torch.tensor([gap], dtype=torch.float32)))
    return sequences

## 2. Improved Transformer Model Architecture

This model uses sinusoidal positional encodings, which is more effective than learned encodings for many tasks. The architecture is a standard transformer encoder stack.

In [ ]:
class PositionalEncoding(nn.Module):
    """Injects positional information into the input sequence."""
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [seq_len, batch_size, d_model]
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

class PrimeGapTransformer(nn.Module):
    def __init__(self, input_dim=1, model_dim=64, nhead=4, num_encoder_layers=3, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.model_dim = model_dim
        self.input_projector = nn.Linear(input_dim, model_dim)
        self.pos_encoder = PositionalEncoding(model_dim, dropout)
        encoder_layers = nn.TransformerEncoderLayer(d_model=model_dim, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=False)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_encoder_layers)
        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, src):
        # src shape: [batch_size, seq_len]
        src = src.unsqueeze(-1)  # Shape: [batch_size, seq_len, 1]
        src = self.input_projector(src) * math.sqrt(self.model_dim)
        # PyTorch Transformer expects [seq_len, batch_size, d_model]
        src = src.permute(1, 0, 2)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src)
        # We only need the output from the last element of the sequence
        output = self.output_layer(output[-1, :, :])
        return output

## 3. Training the Model

Now we'll define our hyperparameters and run the main training loop. We'll use a progress bar from `tqdm` to monitor progress.

In [ ]:
# Hyperparameters
N_PRIMES = 40000  # Generate primes up to 40,000 for more data
SEQ_LENGTH = 20   # Use 20 primes to predict the gap to the 21st
LEARNING_RATE = 0.0005
EPOCHS = 25
BATCH_SIZE = 64

# Generate data
print(f"Generating primes up to {N_PRIMES}...")
primes_list = generate_primes(N_PRIMES)
sequences = create_gap_sequences(primes_list, SEQ_LENGTH)
print(f"Generated {len(primes_list)} primes, creating {len(sequences)} sequences.")

# Model, Loss, and Optimizer
model = PrimeGapTransformer().to(device)
criterion = nn.MSELoss()  # Mean Squared Error is a good choice for gap prediction
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\nStarting training... 🚀")
for epoch in range(EPOCHS):
    model.train()
    random.shuffle(sequences)
    total_loss = 0
    
    # Use tqdm for a nice progress bar
    pbar = tqdm(range(0, len(sequences), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i in pbar:
        batch = sequences[i:i+BATCH_SIZE]
        if not batch:
            continue

        inputs = torch.stack([item[0] for item in batch]).to(device)
        targets = torch.stack([item[1] for item in batch]).to(device)

        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, targets.squeeze(1))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / (len(sequences) / BATCH_SIZE)
    print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f}")

## 4. Evaluating the Model

Let's see how well our model does. We'll give it a sequence of primes it has never seen as a complete sequence during training and ask it to predict the next prime by predicting the gap.

In [ ]:
model.eval()
with torch.no_grad():
    # Pick a random starting point in the prime list for our test
    test_start_index = random.randint(0, len(primes_list) - SEQ_LENGTH - 2)
    
    # Get the input sequence and the last prime within it
    test_sequence = primes_list[test_start_index : test_start_index + SEQ_LENGTH]
    last_prime_in_sequence = test_sequence[-1]
    
    # Find the actual next prime and the true gap
    actual_next_prime = primes_list[test_start_index + SEQ_LENGTH]
    actual_gap = actual_next_prime - last_prime_in_sequence
    
    # Prepare the tensor for the model
    test_tensor = torch.tensor(test_sequence, dtype=torch.float32).unsqueeze(0).to(device)

    # Predict the gap
    predicted_gap = model(test_tensor).item()
    
    # Calculate the predicted prime
    predicted_prime = last_prime_in_sequence + predicted_gap

    print("--- Model Evaluation ---")
    print(f"Input sequence ends with: ..., {last_prime_in_sequence}")
    print(f"Actual next prime: {actual_next_prime} (a gap of {actual_gap})")
    print(f"Model predicted a gap of: {predicted_gap:.2f}")
    print(f"Resulting predicted prime: {predicted_prime:.2f}")
    print(f"Error: {abs(actual_next_prime - predicted_prime):.2f}")